---
# TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 11 dimulai**.
> **Sifat tugas:** individu.

## Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas10_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` (jangan lupa `spark.conf.set("spark.sql.shuffle.partitions", "4")`), lalu kerjakan bagian **A sampai E** berikut:

---

**Inisialisasi SparkSession & Impor Library**

In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum as spark_sum, count, window
import os, shutil, time, pandas as pd
from datetime import datetime

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas10_KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


---

**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

In [47]:
# A. Menyiapkan Streaming Source
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import os, shutil

# 1. Definisikan skema eksplisit
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

# 2. Buat folder simulasi
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# 3. Definisikan readStream (Gunakan file:// untuk lokal)
path_tugas_absolut = "file://" + os.path.abspath(folder_tugas)

# PERHATIKAN: Di sini saya menggunakan nama variabel 'df_stream' agar sesuai dengan gambar Anda
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(path_tugas_absolut)

# 4. Menampilkan status (Ini adalah kode untuk menghasilkan output di gambar)
print("Tugas A: Skema & readStream siap. df_stream.isStreaming:", df_stream.isStreaming)

Tugas A: Skema & readStream siap. df_stream.isStreaming: True


**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

In [54]:
# B. Agregasi per Cabang
from pyspark.sql.functions import col, sum as spark_sum, count

# 1. Tambahkan kolom turunan total_harga (qty x harga_satuan)
# (Menggunakan df_stream dari Poin A)
df_stream_b = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# 2. Buat agregasi groupBy nama_cabang
agregasi_cabang_2 = df_stream_b.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# 3. Jalankan sebagai streaming query dengan sink 'memory'
# Simpan objeknya ke dalam variabel 'query_cabang'
query_cabang = agregasi_cabang_2.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang_2") \
    .start()

# 4. Tampilkan status (Ini kode untuk menghasilkan output di gambar)
print("Tugas B: Query agregasi_cabang berjalan. Status:", query_cabang.status)

Tugas B: Query agregasi_cabang berjalan. Status: {'message': 'Getting offsets from FileStreamSource[file:/home/vboxuser/praktikum-bigdata/tugas10_stream]', 'isDataAvailable': False, 'isTriggerActive': True}


**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

In [55]:
# C. Agregasi Window Waktu (2 Menit)
from pyspark.sql.functions import col, sum as spark_sum, window

# 1. Definisikan ulang readStream dari folder yang sama (tugas10_stream)
# Ini diperbolehkan, satu sumber data bisa dibaca banyak query
df_stream_c = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(path_tugas_absolut)

# 2. Tambahkan kembali kolom total_harga (qty x harga_satuan)
df_stream_c = df_stream_c.withColumn("total_harga", col("qty") * col("harga_satuan"))

# 3. Buat agregasi berbasis Window Waktu 2 Menit
agregasi_waktu_3 = df_stream_c.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan_window")
)

# 4. Jalankan streaming query dengan sink 'memory'
query_waktu = agregasi_waktu_3.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu_3") \
    .start()

# 5. Tampilkan status
print("Tugas C: Query agregasi_waktu berjalan. Status:", query_waktu.status)

Tugas C: Query agregasi_waktu berjalan. Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [52]:
# Cek query yang sedang aktif
for q in spark.streams.active:
    print(f"Nama: {q.name}, Aktif: {q.isActive}, Status: {q.status}")

Nama: agregasi_waktu_3, Aktif: True, Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}
Nama: agregasi_cabang_2, Aktif: True, Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


In [58]:
import pandas as pd
import time
from datetime import datetime

waktu_sekarang = datetime.now()
format_waktu = waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")

# ================= BATCH 1 =================
print("\n--- MENGIRIM BATCH 1 ---")
pd.DataFrame({
    "pesanan_id": ["P1", "P2"],
    "nama_cabang": ["Magelang", "Solo"],
    "nama_menu": ["Kopi Susu", "Americano"],
    "qty": [2, 1],
    "harga_satuan": [20000, 25000],
    "waktu_pesan": [format_waktu, format_waktu]
}).to_csv(f"{folder_tugas}/tugas_batch1.csv", index=False)

time.sleep(10) # Perpanjang waktu tunggu menjadi 10 detik
print("Hasil Agregasi Cabang (Batch 1):")
spark.sql("SELECT * FROM agregasi_cabang_2 ORDER BY total_pendapatan DESC").show()


# ================= BATCH 2 =================
print("\n--- MENGIRIM BATCH 2 ---")
pd.DataFrame({
    "pesanan_id": ["P3", "P4"],
    "nama_cabang": ["Magelang", "Yogyakarta"],
    "nama_menu": ["Latte", "Kopi Susu"],
    "qty": [3, 2],
    "harga_satuan": [30000, 20000],
    "waktu_pesan": [format_waktu, format_waktu]
}).to_csv(f"{folder_tugas}/tugas_batch2.csv", index=False)

time.sleep(10)
print("Hasil Agregasi Cabang (Batch 2):")
spark.sql("SELECT * FROM agregasi_cabang_2 ORDER BY total_pendapatan DESC").show()


# ================= BATCH 3 =================
print("\n--- MENGIRIM BATCH 3 ---")
pd.DataFrame({
    "pesanan_id": ["P5"],
    "nama_cabang": ["Solo"],
    "nama_menu": ["Espresso"],
    "qty": [5],
    "harga_satuan": [18000],
    "waktu_pesan": [format_waktu]
}).to_csv(f"{folder_tugas}/tugas_batch3.csv", index=False)

time.sleep(10)
print("Hasil Akhir Agregasi Cabang (Batch 3):")
spark.sql("SELECT * FROM agregasi_cabang_2 ORDER BY total_pendapatan DESC").show()

print("\nHasil Agregasi Window Waktu (Poin C):")
spark.sql("SELECT window.start, window.end, total_pendapatan_window FROM agregasi_waktu_3").show(truncate=False)


--- MENGIRIM BATCH 1 ---
Hasil Agregasi Cabang (Batch 1):
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          130000|             2|
| Yogyakarta|           40000|             1|
|       Solo|           25000|             1|
+-----------+----------------+--------------+


--- MENGIRIM BATCH 2 ---
Hasil Agregasi Cabang (Batch 2):
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          130000|             2|
| Yogyakarta|           40000|             1|
|       Solo|           25000|             1|
+-----------+----------------+--------------+


--- MENGIRIM BATCH 3 ---
Hasil Akhir Agregasi Cabang (Batch 3):
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          130000|   

**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [35]:
import os
import shutil

# Bersihkan folder output dan checkpoint lama (jika ada)
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# --- PENGAMAN ---
for q in spark.streams.active:
    if q.name == "query_parquet":
        q.stop()
        print("Query 'query_parquet' lama dihentikan.")

# 1. Buat query penulisan ke Parquet
# Catatan: df_stream adalah variabel dari Poin A (data mentah tanpa agregasi)
query_parquet = df_stream.writeStream \
    .format("parquet") \
    .option("path", "file://" + os.path.abspath("tugas10_output")) \
    .option("checkpointLocation", "file://" + os.path.abspath("tugas10_checkpoint")) \
    .outputMode("append") \
    .queryName("query_parquet") \
    .start()

print("Query E (Sink Parquet) berjalan.")

# 2. Kirim satu batch data baru (BATCH 4)
print("\n--- MENGIRIM BATCH 4 (Untuk Parquet) ---")
pd.DataFrame({
    "pesanan_id": ["P6", "P7"],
    "nama_cabang": ["Yogyakarta", "Magelang"],
    "nama_menu": ["Cappuccino", "Kopi Hitam"],
    "qty": [1, 4],
    "harga_satuan": [35000, 15000],
    "waktu_pesan": [format_waktu, format_waktu]
}).to_csv(f"{folder_tugas}/tugas_batch4.csv", index=False)

time.sleep(6)

# 3. Verifikasi output Parquet
print("\nVerifikasi Struktur Folder Parquet:")
!ls tugas10_output

# 4. Baca kembali Parquet sebagai DataFrame Batch
print("\nMembaca kembali Parquet sebagai DataFrame Batch:")
df_parquet_baca_ulang = spark.read.parquet("file://" + os.path.abspath("tugas10_output"))
df_parquet_baca_ulang.show()

Query 'query_parquet' lama dihentikan.
Query E (Sink Parquet) berjalan.

--- MENGIRIM BATCH 4 (Untuk Parquet) ---

Verifikasi Struktur Folder Parquet:
part-00000-85c5d7bc-3007-48ff-9b83-ae0e8460a911-c000.snappy.parquet
part-00001-8bc90976-71e2-435f-8137-fbb37fa293a4-c000.snappy.parquet
part-00002-d2a25a40-96c5-4ac3-813d-51ef5b3ede15-c000.snappy.parquet
part-00003-c9db9b62-f660-44bc-89e9-f842d3790935-c000.snappy.parquet
_spark_metadata

Membaca kembali Parquet sebagai DataFrame Batch:
+----------+-----------+----------+---+------------+-------------------+
|pesanan_id|nama_cabang| nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+----------+---+------------+-------------------+
|        P6| Yogyakarta|Cappuccino|  1|       35000|2026-11-15 21:41:44|
|        P7|   Magelang|Kopi Hitam|  4|       15000|2026-11-15 21:41:44|
|        P3|   Magelang|     Latte|  3|       30000|2026-11-15 21:41:44|
|        P4| Yogyakarta| Kopi Susu|  2|       20000|2026-11-15 21:41:44|


In [36]:
!ls tugas10_output

part-00000-85c5d7bc-3007-48ff-9b83-ae0e8460a911-c000.snappy.parquet
part-00001-8bc90976-71e2-435f-8137-fbb37fa293a4-c000.snappy.parquet
part-00002-d2a25a40-96c5-4ac3-813d-51ef5b3ede15-c000.snappy.parquet
part-00003-c9db9b62-f660-44bc-89e9-f842d3790935-c000.snappy.parquet
_spark_metadata


In [59]:
# Menghentikan semua streaming query yang aktif
for q in spark.streams.active:
    print(f"Menghentikan query: {q.name}")
    q.stop()

# Menutup SparkSession
spark.stop()
print("Semua query telah dihentikan dan SparkSession ditutup.")

Menghentikan query: agregasi_cabang_2
Menghentikan query: agregasi_waktu_3
Semua query telah dihentikan dan SparkSession ditutup.


# Eksplorasi

In [61]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import (
    window, col, sum as spark_sum, count, avg, max as spark_max, min as spark_min, expr
)
import os
import shutil
import time
import pandas as pd
from datetime import datetime

# ==========================================
# 1. INISIALISASI SPARK SESSION & FOLDER
# ==========================================
spark = SparkSession.builder \
    .appName("Eksplorasi-Streaming-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_eksplorasi = "eksplorasi_stream"
shutil.rmtree(folder_eksplorasi, ignore_errors=True)
os.makedirs(folder_eksplorasi, exist_ok=True)

# Skema data transaksi
skema_eksplorasi = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# PERBAIKAN: Menggunakan absolute path dengan file:// untuk membaca folder lokal
path_eksplorasi_absolut = "file://" + os.path.abspath(folder_eksplorasi)

# Membaca Data Streaming
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_eksplorasi) \
    .option("header", True) \
    .load(path_eksplorasi_absolut) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

print("=== STARTING EKSPLORASI STREAMING PIPELINE ===")

# ==========================================
# 2. STRUKTUR TRANSFORMASI & QUERY
# ==========================================

# A. Stream 1: Agregasi Lengkap per Cabang (Sum, Avg, Min, Max)
agregasi_lengkap = df_stream.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("total_transaksi"),
    avg("total_harga").alias("rata_rata_transaksi"),
    spark_max("total_harga").alias("transaksi_terbesar"),
    spark_min("total_harga").alias("transaksi_terkecil")
)

query_agregasi = agregasi_lengkap.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("tbl_agregasi_cabang") \
    .start()

# B. Stream 2: Deteksi Transaksi Besar / High-Value Orders (>= Rp 50.000)
transaksi_besar = df_stream.filter(col("total_harga") >= 50000)

query_high_value = transaksi_besar.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("tbl_high_value_orders") \
    .start()

# C. Stream 3: Windowing Waktu 1 Menit
agregasi_window = df_stream.groupBy(
    window(col("waktu_pesan"), "1 minute")
).agg(
    spark_sum("total_harga").alias("pendapatan_menit"),
    count("pesanan_id").alias("pesanan_menit")
)

query_window = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("tbl_window_menit") \
    .start()

# ==========================================
# 3. SIMULASI ALIRAN DATA (MULTI-BATCH)
# ==========================================

print("\n--- Mengirimkan Batch 1 (Data Transaksi Awal) ---")
pd.DataFrame({
    "pesanan_id": ["E001", "E002", "E003", "E004"],
    "nama_cabang": ["Magelang", "Solo", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Tubruk", "Manual Brew V60", "Es Kopi Aren", "Croissant"],
    "qty": [1, 2, 1, 3],
    "harga_satuan": [15000, 30000, 22000, 25000],  # Total: 15k, 60k (HV), 22k, 75k (HV)
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 4
}).to_csv(f"{folder_eksplorasi}/batch_eksplorasi_1.csv", index=False)

time.sleep(6) # Beri waktu Spark memproses

print("\n[ANALISIS BATCH 1]")
print("1. Ringkasan Performa Cabang:")
spark.sql("""
    SELECT nama_cabang, total_pendapatan, total_transaksi, 
           ROUND(rata_rata_transaksi, 2) AS avg_belanja, 
           transaksi_terbesar
    FROM tbl_agregasi_cabang 
    ORDER BY total_pendapatan DESC
""").show()

print("2. Transaksi Bernilai Tinggi (>= Rp 50.000) yang Terdeteksi Real-Time:")
spark.sql("SELECT pesanan_id, nama_cabang, nama_menu, total_harga FROM tbl_high_value_orders").show()

print("\n--- Mengirimkan Batch 2 (Data Tambahan dengan Delay Waktu) ---")
pd.DataFrame({
    "pesanan_id": ["E005", "E006", "E007"],
    "nama_cabang": ["Solo", "Yogyakarta", "Magelang"],
    "nama_menu": ["Avocado Coffee", "Matcha Latte", "Espresso Double"],
    "qty": [2, 2, 2],
    "harga_satuan": [28000, 26000, 18000], # Total: 56k (HV), 52k (HV), 36k
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder_eksplorasi}/batch_eksplorasi_2.csv", index=False)

time.sleep(6)

print("\n[ANALISIS BATCH 2 - SETELAH AKUMULASI]")
print("1. Ringkasan Performa Cabang Terkini:")
spark.sql("""
    SELECT nama_cabang, total_pendapatan, total_transaksi, 
           ROUND(rata_rata_transaksi, 2) AS avg_belanja
    FROM tbl_agregasi_cabang 
    ORDER BY total_pendapatan DESC
""").show()

print("2. Total Notifikasi High-Value Orders Terkumpul:")
spark.sql("SELECT pesanan_id, nama_cabang, nama_menu, total_harga FROM tbl_high_value_orders").show()

print("3. Performa Penjualan Berdasarkan Window Waktu 1 Menit:")
spark.sql("""
    SELECT window.start, window.end, pendapatan_menit, pesanan_menit 
    FROM tbl_window_menit 
    ORDER BY window.start ASC
""").show(truncate=False)

# ==========================================
# 4. CLEANUP / PEMBERSIHAN QUERY
# ==========================================
query_agregasi.stop()
query_high_value.stop()
query_window.stop()
spark.stop()

print("=== EKSPLORASI SELESAI: Seluruh Streaming Query & SparkSession Dihentikan ===")

=== STARTING EKSPLORASI STREAMING PIPELINE ===

--- Mengirimkan Batch 1 (Data Transaksi Awal) ---

[ANALISIS BATCH 1]
1. Ringkasan Performa Cabang:
+-----------+----------------+---------------+-----------+------------------+
|nama_cabang|total_pendapatan|total_transaksi|avg_belanja|transaksi_terbesar|
+-----------+----------------+---------------+-----------+------------------+
|   Magelang|           90000|              2|    45000.0|             75000|
|       Solo|           60000|              1|    60000.0|             60000|
| Yogyakarta|           22000|              1|    22000.0|             22000|
+-----------+----------------+---------------+-----------+------------------+

2. Transaksi Bernilai Tinggi (>= Rp 50.000) yang Terdeteksi Real-Time:
+----------+-----------+---------------+-----------+
|pesanan_id|nama_cabang|      nama_menu|total_harga|
+----------+-----------+---------------+-----------+
|      E002|       Solo|Manual Brew V60|      60000|
|      E004|   Magelan